# Lecture 3 Lab — Machine Learning Basics
## Train your first neural networks, then build a text classifier

In this lab you will:
1. Train a neural network to **learn an economic relationship** from data (PyTorch).
2. Use a network to **predict the 10-year Treasury yield** from macro indicators.
3. **Build it yourself:** turn **unstructured central-bank text** into numbers and train a model to read it as **hawkish vs. dovish**.

> **Prerequisite:** finish `Lec01_02_Lab_Getting_Started.ipynb` first — it installs Python + PyTorch with China-friendly mirrors. If a package is missing, install it via the Tsinghua mirror, e.g.
> `pip install -i https://pypi.tuna.tsinghua.edu.cn/simple scikit-learn`

*An economist's view:* a neural network is just a **non-linear, non-parametric regression** — we let the computer *find* the functional form instead of assuming it.

In [ ]:
# Quick environment check
import torch, numpy, pandas, matplotlib
import sklearn
torch.set_num_threads(1)   # tiny models train fastest single-threaded (avoids thread oversubscription)
print("torch", torch.__version__, "| sklearn", sklearn.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("✅ ready for the Lecture 3 lab")

import numpy as np
# ReLU helper reused across Part 1B (defined here so later cells work even if the
# JS-animation cell that originally defined it is skipped).
relu = lambda z: np.maximum(0.0, z)


# Part 1 — A neural network is just flexible regression

Instead of assuming $c = \beta_0 + \beta_1 w$, we let a network *find* the shape $c=f(w)$. We train it with the standard loop — forward → loss → `backward()` → `step()` — using **AdamW**, the course-default optimizer.

In [ ]:
%matplotlib inline
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(42)

# A hidden "economic law": consumption c = ln(w) + 0.2 w + noise
N = 400
w = torch.rand(N, 1) * 10 + 1
c = torch.log(w) + 0.2 * w + 0.1 * torch.randn(N, 1)

class SimpleNet(nn.Module):
    def __init__(self, hidden=32, act=nn.ReLU):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, hidden), act(), nn.Linear(hidden, 1))
    def forward(self, x):
        return self.net(x)

model = SimpleNet(hidden=32)
opt   = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-2)  # course default
lossf = nn.MSELoss()

losses = []
for epoch in range(800):
    pred = model(w)
    loss = lossf(pred, c)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

ww = torch.linspace(1, 11, 100).reshape(-1, 1)
with torch.no_grad():
    cc = model(ww)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(w, c, s=8, alpha=0.3, label="data")
ax[0].plot(ww, cc, "r-", lw=2, label="NN fit")
ax[0].plot(ww, np.log(ww) + 0.2 * ww, "k--", label="true law")
ax[0].set_xlabel("wealth w"); ax[0].set_ylabel("consumption c")
ax[0].set_title("The network learns c = ln(w) + 0.2w"); ax[0].legend()
ax[1].plot(losses); ax[1].set_yscale("log")
ax[1].set_xlabel("epoch"); ax[1].set_title("Training loss (log scale)")
plt.tight_layout(); plt.show()
print("final training loss:", round(losses[-1], 4))

> **✏️ Your turn:** in the cell above, set `hidden=4` (too small) and then `hidden=128`; also try `SimpleNet(hidden=32, act=nn.Tanh)`. Re-run and watch the fit and the loss curve. Which choices under- or over-fit?

# Part 1B — How a neural network approximates *any* function

Part 1 showed a network *learning* a shape from data. But **why** can a network fit
essentially any shape? This part opens the black box.

**The big idea — projection onto a basis.** A continuous function lives in an
*infinite-dimensional* space. We cannot store infinitely many numbers, so we **project** it
onto a **finite set of building-block functions** and keep only the coefficients. Fourier uses
sines, splines use local polynomials — a ReLU network uses **shifted ReLU rays**:

$$\hat f(x) = a_0 + \sum_{i=1}^{H} a_i \,\mathrm{ReLU}(w_i x + b_i).$$

**Why ReLU is the right building block.** Look at any target curve: it *turns* (its slope
changes) at certain places, and between turns it has *different slopes*. If you knew **where the
turns are** and **what slope each turn adds**, you would know the whole function. A single ReLU
unit $a\,\mathrm{ReLU}(wx+b)$ encodes exactly that:

- its **kink location** $x = -b/w$ says *where the turn happens*, and
- its **coefficient** $a$ (with $w$) says *how much slope the turn adds*, and in which direction.

Stack enough of them and you can place a turn wherever the target needs one.

**What the *network* adds.** Classical splines fix the knot locations on a grid in advance
(like a Chebyshev grid). A network **learns** $w_i, b_i$ by gradient descent — it *discovers*
where to put the kinks. And note: each ReLU basis function is a **global ray** (nonzero over an
entire half-line), **not** a locally-supported piecewise patch. In higher dimensions the story is
identical — a kink becomes a fold along a hyperplane.

The three slides *ReLU Approximation: Increasing Complexity → More Kink Points → High Fidelity*
are exactly the experiment below.

**Animation 1 — what the two knobs do.** One building block
$g(x) = a\cdot\mathrm{ReLU}(x-b)$. First we slide the shift $b$ (the kink moves); then we vary
the scale $a$ through zero (the ray steepens and flips). Press ▶.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

relu = lambda z: np.maximum(0.0, z)
xg = np.linspace(-3, 3, 200)

fig, ax = plt.subplots(figsize=(7, 4.5))
line, = ax.plot([], [], "r-", lw=2.5)
kink, = ax.plot([], [], "ko", ms=7, zorder=5)      # the kink point
ax.axhline(0, color="gray", lw=0.6); ax.axvline(0, color="gray", lw=0.6)
ax.set_xlim(-3, 3); ax.set_ylim(-6, 6); ax.grid(alpha=0.3)
ax.set_xlabel("x"); ax.set_ylabel(r"$g(x)=a\,\mathrm{ReLU}(x-b)$")

# Phase 1: fix a, slide the shift b.  Phase 2: fix b, sweep the scale a through 0.
bs  = np.linspace(-2.0, 2.0, 25)
as_ = np.linspace(2.0, -2.0, 25)
frames = [("slide b", 1.5, b) for b in bs] + [("scale a", a, 0.0) for a in as_]

def update(fr):
    kind, a, b = fr
    line.set_data(xg, a * relu(xg - b))
    kink.set_data([b], [0.0])
    ax.set_title(f"a = {a:+.2f},  b = {b:+.2f}   (sweeping: {kind})")
    return line, kink

anim = animation.FuncAnimation(fig, update, frames=frames, interval=120, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

**Increasing complexity.** Now let the network *learn* the knobs. We fit
$\sin(x)$ on $[-\pi,\pi]$ with $H = 2, 4, 8, 32$ ReLU units. More units = more kinks = higher
fidelity — the *Increasing Complexity → High Fidelity* arc from the slides. (We reuse the
`SimpleNet` class from Part 1: `Linear(1,H) → ReLU → Linear(H,1)`.)

In [ ]:
import torch, torch.nn as nn

# Arbitrary continuous target to approximate
f    = lambda x: np.sin(x)
x_np = np.linspace(-np.pi, np.pi, 200).reshape(-1, 1)
x_t  = torch.tensor(x_np, dtype=torch.float32)
y_t  = torch.tensor(f(x_np), dtype=torch.float32)

def train_approx(H, epochs=2500, lr=1e-2, seed=0):
    # Fit sin(x) with an H-unit ReLU network; returns (model, final MSE).
    torch.manual_seed(seed)
    net   = SimpleNet(hidden=H)                                   # reused from Part 1
    opt   = torch.optim.AdamW(net.parameters(), lr=lr)           # course-default optimizer
    lossf = nn.MSELoss()
    for _ in range(epochs):
        loss = lossf(net(x_t), y_t)
        opt.zero_grad(); loss.backward(); opt.step()
    return net, loss.item()

xx = x_np.ravel()
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, H in zip(axes.ravel(), [2, 4, 8, 32]):
    net, mse = train_approx(H)
    with torch.no_grad():
        yhat = net(x_t).numpy().ravel()
    ax.plot(xx, f(xx), "b-",  lw=2,  label="target sin(x)")
    ax.plot(xx, yhat,  "r--", lw=2,  label="ReLU network")
    ax.set_title(f"H = {H} ReLU units   (MSE = {mse:.4f})")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.suptitle("More ReLU units  →  more kinks  →  higher fidelity", fontsize=13)
plt.tight_layout(); plt.show()

**Opening the box: the learned ReLU basis.** Take the $H=8$ network and split
its output into the individual contributions $a_i\,\mathrm{ReLU}(w_i x + b_i)$. Each colored line
is **one global ReLU ray** with its own **learned kink** $x=-b_i/w_i$ and slope. Their sum is the
red approximation — this is the *combined* view from the slides.

In [ ]:
H = 8
net, mse = train_approx(H, seed=1)

# Pull the learned parameters out of the two linear layers
W1 = net.net[0].weight.detach().numpy().ravel()   # (H,)  input weights w_i
b1 = net.net[0].bias.detach().numpy().ravel()      # (H,)  input biases  b_i
W2 = net.net[2].weight.detach().numpy().ravel()    # (H,)  output weights a_i
b2 = float(net.net[2].bias.detach().numpy())       # scalar bias a_0

# contribution of unit i:  a_i * ReLU(w_i * x + b_i)
contribs = np.stack([W2[i] * relu(W1[i] * xx + b1[i]) for i in range(H)])   # (H, len(xx))
approx   = contribs.sum(axis=0) + b2

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(xx, f(xx),  "b-",  lw=2.5, label="target sin(x)")
ax.plot(xx, approx, "r--", lw=2,   label="ReLU approximation (sum of rays)")
for i in range(H):
    ax.plot(xx, contribs[i], lw=1.2, alpha=0.75)
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title(f"H = {H}: each colored line is one GLOBAL ReLU ray (not a local spline patch)")
ax.grid(alpha=0.3); ax.legend()
plt.show()

print("learned kink locations  x = -b_i / w_i :", np.round(np.sort(-b1 / W1), 2))

**Animation 2 — combine building blocks to fit the curve.** Start from the
constant $a_0$ and add the learned rays one at a time (left to right by kink location). Watch the
running sum snap toward $\sin(x)$: *combine enough ReLUs and you approximate any function.*

In [ ]:
order = np.argsort(-b1 / W1)     # add rays left-to-right by kink location

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(xx, f(xx), "b-", lw=2.5, label="target sin(x)")
appx, = ax.plot([], [], "r--", lw=2, label="partial sum")
newb, = ax.plot([], [], color="gray", lw=1.2, alpha=0.6, label="ray just added")
ax.set_xlim(xx.min(), xx.max()); ax.set_ylim(-1.7, 1.7)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.grid(alpha=0.3); ax.legend(loc="upper left")

def update2(k):
    run = np.full_like(xx, b2, dtype=float)      # start from the bias a_0
    for j in range(k):
        i = order[j]
        run = run + W2[i] * relu(W1[i] * xx + b1[i])
    appx.set_data(xx, run)
    if k == 0:
        newb.set_data([], [])
    else:
        i = order[k - 1]
        newb.set_data(xx, W2[i] * relu(W1[i] * xx + b1[i]))
    ax.set_title(f"Combining ReLU units:  {k} / {H} added")
    return appx, newb

anim2 = animation.FuncAnimation(fig, update2, frames=H + 1, interval=550, blit=False)
plt.close(fig)
HTML(anim2.to_jshtml())

> **✏️ Your turn:**
> 1. In the increasing-complexity cell, change the list `[2, 4, 8, 32]` — try `H = 1` and
>    `H = 128`. How many kinks do you *need* before the fit looks good?
> 2. Swap the target: set `f = lambda x: np.abs(x)`, or `lambda x: np.tanh(5*x)` (a near-step),
>    or `lambda x: x**2`. Re-run and re-decompose — watch the **kinks relocate** to wherever the
>    new function turns.
> 3. Replace the activation: `SimpleNet(hidden=H, act=nn.Tanh)`. The basis functions become
>    **smooth** instead of kinked — same projection idea, different building block.

### Where does the smoothness come from? *Implicit regularization*

The $H=32$ network had far more parameters than it needed, yet its fit stayed **smooth** instead
of wiggling through every point. Why? We never added a penalty for wiggliness. The answer is
**implicit regularization** — a bias supplied not by the loss, but by **gradient descent itself**.

When a model is *overparameterized*, **infinitely many** parameter settings fit the data equally
well. Nothing in the loss chooses among them — but the optimizer does. Starting from small/zero
initialization, gradient descent moves as little as possible to fit the data, so for a model that
is **linear in its parameters** it provably converges to the **minimum-$\lVert w\rVert_2$** (L2-norm)
solution — the pseudoinverse solution `np.linalg.pinv` returns. **Yes — it is exactly the L2 norm**,
even though no L2 penalty was ever written down. And a smaller weight norm means a **smoother
function**, which is the smoothness we saw.

The cell below *demonstrates* this in two panels — and does not just assert it:

- **Left — the geometry.** One data equation, two weights ($w_1+w_2=1$): the solutions form a
  whole line. The gradient is always parallel to $(1,1)$, so GD from the origin can only move along
  that direction and **stops at the point of the line closest to the origin** — the
  minimum-$\lVert w\rVert_2$ solution $(0.5, 0.5)$. Any other interpolant (e.g. $(1,0)$) has a
  larger norm.
- **Right — the same bias in function space.** Fit **7 points** of $\sin$ with **41 weights**
  (40 ReLU features + a bias) — wildly overparameterized, so infinitely many exact fits exist.
  We compute three of them and compare:
    1. `a_min` — the analytic **minimum-norm** (pseudoinverse) solution;
    2. `a_gd` — the weights **plain gradient descent from zero actually reaches**;
    3. `a_big` — a deliberately **large-norm** interpolant (we add a null-space direction).

  The punchline is printed: $\lVert a_{gd}-a_{min}\rVert \approx 10^{-4}$ — **GD reproduces the
  minimum-norm solution on its own**, with no penalty term anywhere. Its curve (magenta) lies right
  on top of the min-norm curve (green) and is smooth; `a_big` fits the *same 7 points* but wiggles
  violently between them.

*(We use a fixed ReLU **feature** basis so the model is linear in the weights and "minimum L2 norm"
holds exactly. A trained network is the nonlinear analogue — the bias is toward small-norm, smooth
solutions, though the exact statement is subtler; see the slides, Lec.\ 3 T3.)*

In [ ]:
# ============================================================================
#  Implicit regularization  =  gradient descent's hidden bias toward min ||w||_2
#  numpy only; the model is LINEAR in its weights, so "minimum L2 norm" is EXACT
#  and we can *demonstrate* it (not just assert it).
# ============================================================================

# ---------------------------------------------------------------------------
# PANEL A -- the geometry, in 2 dimensions you can actually see.
#   One data equation, two unknown weights:      w1 + w2 = 1
#   => the solutions form a whole LINE; every point on it fits the data perfectly.
#   Which point does gradient descent return, starting from the origin?
# ---------------------------------------------------------------------------
w = np.array([0.0, 0.0]); traj = [w.copy()]
for _ in range(60):
    grad = (w.sum() - 1.0) * np.array([1.0, 1.0])     # d/dw of  0.5*(w1 + w2 - 1)^2
    w = w - 0.15 * grad                               # one gradient-descent step
    traj.append(w.copy())
traj = np.array(traj)

# The gradient is ALWAYS parallel to (1,1), so from the origin GD can only travel
# along that direction -> it halts at the point of the line closest to the origin,
# i.e. the minimum-||w|| solution (0.5, 0.5).  Any other interpolant is larger.
print("PANEL A  -- every point on  w1+w2=1  fits the data; GD selects the smallest")
print("   GD endpoint         :", np.round(traj[-1], 3), "  ||w|| = %.3f" % np.linalg.norm(traj[-1]))
print("   another interpolant : [1. 0.]       ||w|| = %.3f  (larger!)" % np.linalg.norm([1, 0]))
print()

# ---------------------------------------------------------------------------
# PANEL B -- the same bias in function space (this is WHY the fit came out smooth).
#   Fit 7 points of sin(x) with 40 shifted-ReLU features + 1 bias = 41 weights.
#   41 weights but only 7 equations -> massively over-parameterized: infinitely
#   many weight vectors fit the 7 points EXACTLY.  We build three of them.
# ---------------------------------------------------------------------------
xd = np.linspace(-np.pi, np.pi, 7); yd = np.sin(xd)   # 7 training points
shifts = np.linspace(-np.pi, np.pi, 40)               # 40 fixed ReLU kink locations
def feats(x):                                         # design matrix of ReLU features
    x = np.atleast_1d(x)[:, None]
    return np.concatenate([np.maximum(0, x - shifts[None, :]), np.ones((len(x), 1))], axis=1)
Phi = feats(xd)                                       # shape (7, 41):  #weights >> #data

# (1) analytic MINIMUM-norm interpolant = the pseudoinverse solution
a_min = np.linalg.pinv(Phi) @ yd

# (2) DEMONSTRATE the implicit bias: run plain full-batch gradient descent from ZERO
#     on the very same linear model, with NO penalty, and watch where it lands.
a  = np.zeros(Phi.shape[1])
lr = 0.5 / np.linalg.norm(Phi, 2) ** 2                # safe step = 0.5 / largest curvature
for _ in range(120_000):
    a = a - lr * (Phi.T @ (Phi @ a - yd))             # gradient of 0.5*||Phi a - y||^2
a_gd = a

# (3) a deliberately LARGE-norm interpolant: add a null-space direction of Phi.
#     Phi @ (null vector) = 0, so it still fits all 7 points -- but ||a|| balloons
#     and the curve wiggles between the data points.
_, S, Vt = np.linalg.svd(Phi)
a_big = a_min + 6.0 * Vt[len(S):][0]                  # rows of Vt past rank = null space

print("PANEL B  -- 41 weights, only 7 data points  ->  infinitely many perfect fits")
print("   ||a_min||  (pseudoinverse)              = %.4f" % np.linalg.norm(a_min))
print("   ||a_gd||   (gradient descent from zero) = %.4f" % np.linalg.norm(a_gd))
print("   ||a_gd - a_min||                        = %.2e   <-- GD FOUND the min-norm solution!"
      % np.linalg.norm(a_gd - a_min))
print("   ||a_big||  (also interpolates, but big) = %.4f" % np.linalg.norm(a_big))
print("   train error:  min %.1e   gd %.1e   big %.1e   (all THREE fit the 7 points exactly)"
      % (np.abs(Phi @ a_min - yd).max(), np.abs(Phi @ a_gd - yd).max(), np.abs(Phi @ a_big - yd).max()))

# ------------------------------------ plots --------------------------------
xg = np.linspace(-np.pi, np.pi, 400)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

# Panel A: the 2-D geometry
ax[0].plot([1, 0], [0, 1], "g-", lw=2, label="all solutions: $w_1+w_2=1$")
ax[0].plot(traj[:, 0], traj[:, 1], "o-", color="crimson", ms=3, label="GD from origin")
ax[0].plot(0.5, 0.5, "k*", ms=16, label="min-norm solution")
ax[0].plot([0, 0.5], [0, 0.5], "k:", lw=1); ax[0].scatter([0], [0], c="blue")
ax[0].annotate("start", (0.03, 0.03)); ax[0].set_aspect("equal")
ax[0].set_xlabel("$w_1$"); ax[0].set_ylabel("$w_2$"); ax[0].grid(alpha=.3); ax[0].legend(fontsize=8)
ax[0].set_title(r"GD $\to$ closest point to origin = min $\|w\|_2$")

# Panel B: the three interpolants (GD sits exactly on min-norm)
ax[1].plot(xg, np.sin(xg), "b-", lw=1, alpha=.4, label="sin(x)")
ax[1].plot(xg, feats(xg) @ a_gd,  "m-",  lw=4, alpha=.4, label=r"GD fit ($\|a\|$=%.1f)" % np.linalg.norm(a_gd))
ax[1].plot(xg, feats(xg) @ a_min, "g--", lw=1.6, label=r"min-norm fit ($\|a\|$=%.1f)" % np.linalg.norm(a_min))
ax[1].plot(xg, feats(xg) @ a_big, "r:",  lw=2, label=r"large-norm fit ($\|a\|$=%.1f)" % np.linalg.norm(a_big))
ax[1].scatter(xd, yd, c="k", zorder=5, label="7 data points (all fit these!)")
ax[1].set_xlabel("x"); ax[1].grid(alpha=.3); ax[1].legend(fontsize=8)
ax[1].set_title("GD = min-norm = smooth;   large norm = wiggly")
plt.tight_layout(); plt.show()

# Part 2 — Predicting the 10-year Treasury yield

A real-flavored macro task: predict the 10-year yield from the **Fed Funds rate** and **unemployment**. We try to download live **FRED** data and **fall back to a synthetic panel** if you are offline — so the lab always runs.

In [ ]:
import numpy as np, pandas as pd

def load_macro_data():
    # Try live FRED; fall back to a synthetic but plausible panel if offline.
    try:
        import datetime
        import pandas_datareader.data as web
        raw = web.DataReader(["DGS10", "FEDFUNDS", "UNRATE"], "fred",
                             datetime.datetime(1990, 1, 1), datetime.datetime(2024, 1, 1))
        raw = raw.resample("ME").last().dropna()
        raw = raw.rename(columns={"DGS10": "Yield10", "FEDFUNDS": "FedFunds", "UNRATE": "Unemp"})
        return raw[["FedFunds", "Unemp", "Yield10"]].reset_index(drop=True), "FRED (live)"
    except Exception:
        rng = np.random.default_rng(0)
        T = 360
        fed   = np.clip(np.cumsum(rng.normal(0, 0.3, T)) + 3, 0, 12)
        unemp = np.clip(6 + np.sin(np.linspace(0, 12, T)) + rng.normal(0, 0.4, T), 3, 11)
        yld   = 1.5 + 0.8 * fed - 0.15 * unemp + rng.normal(0, 0.4, T)   # stylized term structure
        df = pd.DataFrame({"FedFunds": fed, "Unemp": unemp, "Yield10": yld})
        return df, "synthetic (offline fallback)"

data, source = load_macro_data()
print("Data source:", source, "| shape:", data.shape)
data.head()

In [ ]:
feat_cols = ["FedFunds", "Unemp"]
X = torch.tensor(data[feat_cols].values, dtype=torch.float32)
y = torch.tensor(data[["Yield10"]].values, dtype=torch.float32)

# Standardize (networks train better on centered, unit-variance inputs)
Xm, Xs = X.mean(0), X.std(0); ym, ys = y.mean(0), y.std(0)
Xn, yn = (X - Xm) / Xs, (y - ym) / ys

# Time-series split: NEVER shuffle before splitting a time series
split = int(0.8 * len(Xn))
Xtr_m, Xte_m = Xn[:split], Xn[split:]
ytr_m, yte_m = yn[:split], yn[split:]

class MacroNet(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        return self.net(x)

macro = MacroNet(len(feat_cols))
opt   = torch.optim.AdamW(macro.parameters(), lr=1e-2, weight_decay=1e-2)
lossf = nn.MSELoss()

tr, te = [], []
for epoch in range(1500):
    macro.train()
    loss = lossf(macro(Xtr_m), ytr_m)
    opt.zero_grad(); loss.backward(); opt.step()
    tr.append(loss.item())
    macro.eval()
    with torch.no_grad():
        te.append(lossf(macro(Xte_m), yte_m).item())

with torch.no_grad():
    pred_all = (macro(Xn) * ys + ym).numpy().ravel()

plt.figure(figsize=(10, 4))
plt.plot(data["Yield10"].values, color="black", alpha=0.6, label="actual 10Y yield")
plt.plot(pred_all, color="C0", label="MacroNet prediction")
plt.axvline(split, color="red", ls="--", label="train/test split")
plt.xlabel("month"); plt.ylabel("yield (%)")
plt.title(f"Predicting the 10-year yield  [{source}]")
plt.legend(); plt.grid(alpha=0.3); plt.show()
print(f"train MSE {tr[-1]:.3f} | test MSE {te[-1]:.3f}")

> **✏️ Your turn:** add a third feature (for example a one-month lag of the yield, `data["Yield10"].shift(1)`) or widen the network, and see whether the **test** MSE improves. In time series we never shuffle before splitting — why would that leak information?

# Part 3 — Build it yourself: reading the Fed's words

Lecture 1 motivated ML by its power to **harness unstructured data**. Here you run the full pipeline on text:

**unstructured text → structured features → model → visualize.**

We classify short central-bank sentences as **hawkish** (leaning toward higher rates) or **dovish** (leaning toward easing). We build a small labeled set right here in the notebook — no download needed.

In [ ]:
# A small labeled set of central-bank-style sentences, built from templates
# (transparent and reproducible). In practice you would use real text — see Lab 5B.
subjects = ["The committee", "Federal Reserve officials", "Policymakers",
            "Several members", "The central bank", "The latest statement"]
hawkish_phrases = [
    "signaled further interest rate hikes",
    "favored a more restrictive policy stance",
    "stressed that inflation remains too high",
    "supported raising rates to curb inflation",
    "warned about the risk of overheating",
    "leaned toward additional monetary tightening",
    "emphasized keeping rates higher for longer",
    "pointed to strong demand and rising prices",
]
dovish_phrases = [
    "signaled possible interest rate cuts",
    "favored a more accommodative policy stance",
    "stressed that economic growth is slowing",
    "supported lowering rates to support activity",
    "warned about the risk of a downturn",
    "leaned toward monetary easing",
    "emphasized supporting employment and jobs",
    "pointed to weak demand and cooling prices",
]
hawkish = [f"{s} {p}." for s in subjects for p in hawkish_phrases]
dovish  = [f"{s} {p}." for s in subjects for p in dovish_phrases]

texts  = hawkish + dovish
labels = [1] * len(hawkish) + [0] * len(dovish)   # 1 = hawkish, 0 = dovish
print(f"{len(texts)} sentences ({len(hawkish)} hawkish, {len(dovish)} dovish)")
print("examples:")
print("  HAWKISH:", hawkish[0])
print("  DOVISH :", dovish[0])

### Step 1 — Unstructured → structured (bag of words)

A model cannot read text, only numbers. `CountVectorizer` builds a **vocabulary** and turns each sentence into a row of word counts — the **document-term matrix**.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vec  = CountVectorizer(stop_words="english")
Xbow = vec.fit_transform(texts)
vocab = vec.get_feature_names_out()
print("vocabulary size:", len(vocab))
print("document-term matrix shape:", Xbow.shape, "(sentences x words)")

Let's *see* the conversion: a heatmap of the first few sentences against the most common words. Each row is a sentence, each column a word, each cell a count.

In [ ]:
dense = Xbow.toarray()
top = np.argsort(dense.sum(0))[::-1][:20]          # 20 most frequent words
plt.figure(figsize=(11, 4))
plt.imshow(dense[:8][:, top], aspect="auto", cmap="Blues")
plt.xticks(range(len(top)), vocab[top], rotation=90)
plt.yticks(range(8), [f"sentence {i}" for i in range(8)])
plt.title("Unstructured → structured: the document-term matrix")
plt.colorbar(label="word count"); plt.tight_layout(); plt.show()

### Step 2 — A stylized model (worked baseline)

The simplest classifier — **logistic regression** on the word counts — already learns the hawkish/dovish split. We compare its accuracy to a **majority-class baseline** (always guessing the more common label).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

Xtr_b, Xte_b, ytr_b, yte_b = train_test_split(
    Xbow, labels, test_size=0.3, random_state=0, stratify=labels)

clf = LogisticRegression(max_iter=1000)
clf.fit(Xtr_b, ytr_b)
y_pred = clf.predict(Xte_b)

acc      = accuracy_score(yte_b, y_pred)
baseline = max(np.mean(ytr_b), 1 - np.mean(ytr_b))
print(f"Logistic-regression accuracy: {acc:.2f}   (majority-class baseline: {baseline:.2f})")

### Step 3 — ✏️ Your turn: build a PyTorch classifier

You just saw a one-line scikit-learn baseline. Now **build your own** small neural-network classifier on the same bag-of-words vectors, reusing the training loop from Part 1. A full solution is in the **Appendix** — try it first.

In [ ]:
# ✏️ YOUR TURN — build a small PyTorch MLP classifier on the bag-of-words vectors.
#   1. Dense float tensors:   torch.tensor(Xtr_b.toarray(), dtype=torch.float32)
#   2. Long label tensors:    torch.tensor(ytr_b, dtype=torch.long)
#   3. Model:  nn.Sequential(nn.Linear(n_words, 32), nn.ReLU(), nn.Linear(32, 2))
#   4. Loss = nn.CrossEntropyLoss();  optimizer = torch.optim.AdamW(lr=1e-2, weight_decay=1e-2)
#   5. Train ~300 epochs, then compute test accuracy.
# (Solution in the Appendix.)

# your code here
print("TODO: implement the PyTorch MLP — see the Appendix for one solution.")

### Step 4 — Visualize what the model learned

Two pictures: the **confusion matrix** (where it is right vs. wrong on held-out sentences), and the **words** the model treats as most hawkish vs. most dovish.

In [ ]:
# Confusion matrix
cm = confusion_matrix(yte_b, y_pred)
plt.figure(figsize=(4.5, 4))
plt.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center", fontsize=12)
plt.xticks([0, 1], ["dovish", "hawkish"]); plt.yticks([0, 1], ["dovish", "hawkish"])
plt.xlabel("predicted"); plt.ylabel("actual")
plt.title("Confusion matrix (held-out sentences)")
plt.colorbar(); plt.tight_layout(); plt.show()

# Most hawkish / most dovish words (logistic-regression coefficients)
coef  = clf.coef_[0]
order = np.argsort(coef)
dov, haw = order[:10], order[::-1][:10]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].barh(vocab[dov], coef[dov], color="C0"); ax[0].set_title("Most 'dovish' words"); ax[0].invert_yaxis()
ax[1].barh(vocab[haw], coef[haw], color="C3"); ax[1].set_title("Most 'hawkish' words"); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()

### Make it your own

- Add your own sentences to `hawkish` / `dovish` and re-run — does accuracy hold?
- Swap `CountVectorizer` for `TfidfVectorizer`, or add `ngram_range=(1, 2)` to capture phrases like *"rate cuts"*.
- Scale up: load real **FOMC minutes** (see `Lab5B_fomc_minutes_10y_model.ipynb`) and reuse this exact pipeline.

---
## Appendix — Reference solution (PyTorch classifier)

Try Step 3 yourself first, then compare.

In [ ]:
# --- Solution: a tiny PyTorch MLP classifier on the bag-of-words vectors ---
Xtr_t = torch.tensor(Xtr_b.toarray(), dtype=torch.float32)
Xte_t = torch.tensor(Xte_b.toarray(), dtype=torch.float32)
ytr_t = torch.tensor(ytr_b, dtype=torch.long)
yte_t = torch.tensor(yte_b, dtype=torch.long)

torch.manual_seed(0)
mlp   = nn.Sequential(nn.Linear(Xtr_t.shape[1], 32), nn.ReLU(), nn.Linear(32, 2))
opt   = torch.optim.AdamW(mlp.parameters(), lr=1e-2, weight_decay=1e-2)
lossf = nn.CrossEntropyLoss()

for epoch in range(300):
    logits = mlp(Xtr_t)
    loss   = lossf(logits, ytr_t)
    opt.zero_grad(); loss.backward(); opt.step()

with torch.no_grad():
    mlp_acc = (mlp(Xte_t).argmax(1) == yte_t).float().mean().item()
print(f"PyTorch MLP test accuracy: {mlp_acc:.2f}")

---
## Wrap-up

You trained a neural network to learn an economic law, used one to predict the 10-year yield, and built a full **text → features → model → visualization** pipeline.

**Next:**
- `Lab5B_fomc_minutes_10y_model.ipynb` — the same idea on **real FOMC minutes** (TF-IDF and a pre-trained FinBERT model).
- **Lecture 4** — networks that respect economic structure (convex / monotone) for dynamic models.